# Two-Layer Neural Networks with PyTorch

## What this notebook demonstrates

- CIFAR-10 image classification with PyTorch
- tensor operations and autograd
- manual training loops
- the nn.Module API
- the nn.Sequential API and optimizer workflows

## Academic context

This notebook is a cleaned portfolio version of MSc coursework and implementation practice. It is included to demonstrate foundational computer vision and deep learning skills.


In [ ]:
# Project-local setup
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "engine").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the project root containing the engine/ package.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")


This notebook moves from manual neural-network code toward PyTorch workflows for CIFAR-10 classification. It demonstrates tensors, autograd, module definitions, optimizer APIs, and concise sequential models.


## Why deep learning frameworks matter

Deep learning frameworks make GPU acceleration, automatic differentiation, reusable model components, and optimizer workflows available through concise APIs. PyTorch is used here because it keeps tensor operations explicit while removing the need to manually implement every backward pass.


## Notebook outline

1. CIFAR-10 data preparation.
2. Bare PyTorch tensor operations and autograd.
3. Model definition with `nn.Module`.
4. Compact architectures with `nn.Sequential`.
5. Optimizer, activation, dropout, and scheduler experiments.


This notebook has four parts. You will learn PyTorch on **three different levels of abstraction**, which will help you understand it better and prepare you for the final open-ended challenge.

1. Part I, Preparation: we will use CIFAR-10 dataset.
2. Part II, **Barebones PyTorch**: *Abstraction level 1*, we will work directly with the lowest-level PyTorch Tensors.
3. Part III, **PyTorch Module API**: *Abstraction level 2*, we will use `nn.Module` to define arbitrary neural network architecture.
4. Part IV, **PyTorch Sequential API**: *Abstraction level 3*, we will use `nn.Sequential` to define a linear feed-forward network very conveniently.

Here is a table of comparison:

| API           | Flexibility | Convenience |
|---------------|-------------|-------------|
| Barebone      | High        | Low         |
| `nn.Module`     | High        | Medium      |
| `nn.Sequential` | Low         | High        |


## GPU


If CUDA is available, the setup code uses it automatically; otherwise the notebook runs on CPU.


### Setup


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as datasets
import torchvision.transforms as T

import numpy as np

USE_GPU = True
dtype = torch.float32 # We will be using float throughout this tutorial.

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Constant to control how frequently we print train loss.
print_every = 100
print('using device:', device)


## Part I. Preparation


### Load CIFAR-10 dataset


Now, let's load the CIFAR-10 dataset.

This might take a couple minutes the first time you do it, but the files should stay cached after that.

In previous parts of the earlier notebooks we had to write our own code to download the CIFAR-10 dataset, preprocess it, and iterate through it in minibatches.

To make the whole procedure more efficient, we managed to save the whole dataset in ``.npy`` files.

PyTorch provides some convenient tools to automate this process even more for us.


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Define the mean and standard deviation for CIFAR-10
mean = (0.4914, 0.4822, 0.4465)  # Precomputed mean for CIFAR-10
std = (0.2023, 0.1994, 0.2010)   # Precomputed std for CIFAR-10

# Define the transform for preprocessing and data augmentation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Load the CIFAR-10 dataset
dataset_path = str(DATA_DIR)  
cifar10_dataset = datasets.CIFAR10(root=dataset_path, train=True, download=True, transform=transform) 

# Split the dataset into train, val, and test sets
num_train = 45000  # Number of training examples
num_val = 5000    # Number of validation examples
num_test = 10000   # Number of test examples (CIFAR-10 test set size)

train_dataset, val_dataset = random_split(cifar10_dataset, [num_train, num_val])

test_dataset = datasets.CIFAR10(root=dataset_path, train=False, download=True, transform=transform)

# Define DataLoaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Print dataset sizes for verification
print(f"Train set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")


### Visualize the class distributions


We have now defined our train, validation and test dataloaders. However, it would be good to check the data distribution of each dataloader, in order to check how many examples per class are included. We do this as a sanity check, as we want our dataloaders to be distributed close to uniform. Let's define some helper functions:


In [ ]:
import numpy as np
from collections import Counter

def get_class_distribution(loader, dataset_classes):
    """
    Computes the distribution of classes in a DataLoader.

    Args:
    - loader: DataLoader to inspect
    - dataset_classes: List of class names for the dataset

    Returns:
    - class_counts: Dictionary with class names as keys and counts as values
    """
    class_counts = Counter()
    for inputs, labels in loader:
        class_counts.update(labels.numpy())

    # Convert to a dictionary with class names
    class_distribution = {dataset_classes[i]: class_counts[i] for i in range(len(dataset_classes))}
    return class_distribution

def display_class_distribution(distribution, title="Class Distribution"):
    """
    Displays the class distribution as a bar chart.

    Args:
    - distribution: Dictionary with class names as keys and counts as values
    - title: Title for the bar chart
    """
    import matplotlib.pyplot as plt

    class_names = list(distribution.keys())
    counts = list(distribution.values())

    plt.figure(figsize=(10, 6))
    plt.bar(class_names, counts)
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()


Now, let's visualize the distribution of the examples per class for each dataloader:


In [ ]:
# CIFAR-10 classes
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

# Compute and display class distribution for each DataLoader
for loader, name in [(train_loader, "Train"), (val_loader, "Validation"), (test_loader, "Test")]:
    distribution = get_class_distribution(loader, cifar10_classes)
    print(f"{name} DataLoader Distribution:")
    for cls, count in distribution.items():
        print(f"  {cls}: {count}")
    display_class_distribution(distribution, title=f"{name} DataLoader Class Distribution")


### Visualize CIFAR-10 images


Now let's visualize a few examples from each class in the CIFAR-10 dataset.

By displaying images alongside their corresponding class labels, we can verify the dataset’s contents and ensure the classes are represented correctly.

This visualization helps us understand the data distribution and provides insights into the dataset’s characteristics.


In [ ]:
import matplotlib.pyplot as plt

def visualize_images_per_class(dataset, dataset_classes, num_images=5):
    """
    Visualizes a few images from each class in the dataset.

    Args:
    - dataset: The dataset to sample from.
    - dataset_classes: List of class names for the dataset.
    - num_images: Number of images to display per class.
    """
    class_to_indices = {cls: [] for cls in range(len(dataset_classes))}

    # Group indices by class
    for idx, (_, label) in enumerate(dataset):
        if len(class_to_indices[label]) < num_images:
            class_to_indices[label].append(idx)
        if all(len(indices) == num_images for indices in class_to_indices.values()):
            break

    # Plot the images
    fig, axes = plt.subplots(len(dataset_classes), num_images + 1, figsize=((num_images + 1) * 2, len(dataset_classes) * 2))
    for cls_idx, cls_name in enumerate(dataset_classes):
        indices = class_to_indices[cls_idx]

        # Add class name to the first column of each row
        axes[cls_idx, 0].text(0.5, 0.5, cls_name, fontsize=12, ha='center', va='center', transform=axes[cls_idx, 0].transAxes)
        axes[cls_idx, 0].axis("off")

        for img_idx, data_idx in enumerate(indices):
            img, label = dataset[data_idx]
            ax = axes[cls_idx, img_idx + 1]  # Shift by 1 for the class name column
            img = img.permute(1, 2, 0)  # Convert CHW to HWC
            img = img * torch.tensor(std).view(1, 1, 3) + torch.tensor(mean).view(1, 1, 3)  # De-normalize
            img = torch.clamp(img, 0, 1)  # Clip values to valid range
            ax.imshow(img)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

# Visualize 5 images per class for CIFAR-10 train set
visualize_images_per_class(cifar10_dataset, cifar10_classes, num_images=5)


## Part II. Two-Layer Network with Barebone PyTorch


PyTorch comes with high-level APIs to help us define model architectures conveniently, which we will cover in Part II of this tutorial. In this section, we will start with the **barebone** PyTorch elements to understand the autograd engine better. After this notebook, you will come to appreciate the high-level model API more.

We will start with a simple fully-connected ReLU network with two hidden layers and no biases for CIFAR-10 classification.

This implementation computes the **forward** pass using operations on PyTorch Tensors, and uses PyTorch `autograd` to compute gradients. It is important that you understand every line, because you will write a harder version after the example.

When we create a PyTorch Tensor with `requires_grad=True`, then operations involving that Tensor will not just compute values; they will also build up a **computational graph** in the background, allowing us to easily backpropagate through the graph to compute gradients of some Tensors with respect to a downstream loss. Concretely if x is a Tensor with `x.requires_grad == True` then after backpropagation `x.grad` will be another Tensor holding the gradient of x with respect to the scalar loss at the end.


### PyTorch Tensors: Flatten Function
A PyTorch Tensor is conceptionally similar to a numpy array: it is an n-dimensional grid of numbers and, like numpy, PyTorch provides many functions to efficiently operate on Tensors. As a simple example, we provide a `flatten` function below which reshapes image data for use in a fully-connected neural network.

Recall that image data is typically stored in a Tensor of shape `N x C x H x W`, where:

* `N` is the number of datapoints
* `C` is the number of channels
* `H` is the height of the intermediate feature map in pixels
* `W` is the height of the intermediate feature map in pixels

This is the right way to represent the data when we are doing something like a 2D convolution, that needs **spatial understanding** of where the intermediate features are relative to each other. When we use fully connected affine layers to process the image, however, we want each datapoint to be represented by a **single vector** - it's no longer useful to segregate the different channels, rows, and columns of the data. So, we use a "flatten" operation to collapse the `C x H x W` values per representation into a single long vector. The flatten function below first reads in the N, C, H, and W values from a given batch of data, and then returns a "view" of that data. "View" is analogous to numpy's "reshape" method: it reshapes x's dimensions to be N x ??, where ?? is allowed to be anything (in this case, it will be `C x H x W`, but we don't need to specify that explicitly).


In [ ]:
def flatten(x):
    N = x.shape[0] # read in N, C, H, W
    return x.view(N, -1)  # "flatten" the C * H * W values into a single vector per image

def test_flatten():
    x = torch.arange(12).view(2, 1, 3, 2)
    print('Before flattening: ', x)
    print('After flattening: ', flatten(x))

test_flatten()


### Barebone PyTorch: Two-Layer Network

Here we define a function `two_layer_fc` which performs the **forward** pass of a two-layer fully-connected ReLU network on a batch of image data. After defining the forward pass we check that it doesn't crash and that it produces outputs of the right shape by running zeros through the network.


In [ ]:
import torch.nn.functional as F  # useful stateless functions

def two_layer_fc(x, params):
    """
    A fully-connected neural networks; the architecture is:
    NN is fully connected -> ReLU -> fully connected layer.
    Note that this function only defines the forward pass;
    PyTorch will take care of the backward pass for us.

    The input to the network will be a minibatch of data, of shape
    (N, d1, ..., dM) where d1 * ... * dM = D. The hidden layer will have H units,
    and the output layer will produce scores for C classes.

    Inputs:
    - x: A PyTorch Tensor of shape (N, d1, ..., dM) giving a minibatch of
      input data.
    - params: A list [w1, w2] of PyTorch Tensors giving weights for the network;
      w1 has shape (D, H) and w2 has shape (H, C).

    Returns:
    - scores: A PyTorch Tensor of shape (N, C) giving classification scores for
      the input data x.
    """
    # first we flatten the image
    x = flatten(x)  # shape: [batch_size, C x H x W]

    w1, w2 = params

    # Forward pass: compute predicted y using operations on Tensors. Since w1 and
    # w2 have requires_grad=True, operations involving these Tensors will cause
    # PyTorch to build a computational graph, allowing automatic computation of
    # gradients. Since we are no longer implementing the backward pass by hand we
    # don't need to keep references to intermediate values.
    # you can also use `.clamp(min=0)`, equivalent to F.relu()
    x = F.relu(x.mm(w1))
    x = x.mm(w2)
    return x


def two_layer_fc_test():
    hidden_layer_size = 42
    x = torch.zeros((64, 50), dtype=dtype)  # minibatch size 64, feature dimension 50
    w1 = torch.zeros((50, hidden_layer_size), dtype=dtype)
    w2 = torch.zeros((hidden_layer_size, 10), dtype=dtype)
    scores = two_layer_fc(x, [w1, w2])
    print(scores.size())  # expected shape [64, 10]

two_layer_fc_test()


### Barebone PyTorch: Initialization
Let's write a couple utility methods to initialize the weight matrices for our models.

- `random_weight(shape)` initializes a weight tensor with the Kaiming normalization method.
- `zero_weight(shape)` initializes a weight tensor with all zeros. Useful for instantiating bias parameters.

The `random_weight` function uses the Kaiming normal initialization method, described in:

He et al, *Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification*, ICCV 2015, https://arxiv.org/abs/1502.01852


In [ ]:
def random_weight(shape):
    """
    Create random Tensors for weights; setting requires_grad=True means that we
    want to compute gradients for these Tensors during the backward pass.
    We use Kaiming normalization: sqrt(2 / fan_in)
    """
    if len(shape) == 2:  # FC weight
        fan_in = shape[0]
    else:
        fan_in = np.prod(shape[1:]) # conv weight [out_channel, in_channel, kH, kW]
    # randn is standard normal distribution generator.
    w = torch.randn(shape, device=device, dtype=dtype) * np.sqrt(2. / fan_in)
    w.requires_grad = True
    return w

def zero_weight(shape):
    return torch.zeros(shape, device=device, dtype=dtype, requires_grad=True)

# create a weight of shape [3 x 5]
# expected shape the type `torch.cuda.FloatTensor` if you use GPU.
# Otherwise it should be `torch.FloatTensor`
random_weight((3, 5))


### Barebone PyTorch: Check Accuracy
When training the model we will use the following function to check the accuracy of our model on the training or validation set.

When checking accuracy we don't need to compute any gradients; as a result we don't need PyTorch to build a computational graph for us when we compute scores. To prevent a graph from being built we scope our computation under a `torch.no_grad()` context manager.


In [ ]:
def check_accuracy_part2(loader, model_fn, params):
    """
    Check the accuracy of a classification model.

    Inputs:
    - loader: A DataLoader for the data split we want to check
    - model_fn: A function that performs the forward pass of the model,
      with the signature scores = model_fn(x, params)
    - params: List of PyTorch Tensors giving parameters of the model

    Returns: Nothing, but prints the accuracy of the model
    """
    #split = 'val' if loader.dataset.train else 'test'
    split = 'val' if len(loader) == 79 else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.int64)
            scores = model_fn(x, params)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))


### BareBone PyTorch: Training Loop
We can now set up a basic training loop to train our network. We will train the model using stochastic gradient descent without momentum. We will use `torch.functional.cross_entropy` to compute the loss; you can [read about it here](http://pytorch.org/docs/stable/nn.html#cross-entropy).

The training loop takes as input the neural network function, a list of initialized parameters (`[w1, w2]` in our example), and learning rate.


In [ ]:
def train_part2(model_fn, params, learning_rate):
    """
    Train a model on CIFAR-10.

    Inputs:
    - model_fn: A Python function that performs the forward pass of the model.
      It should have the signature scores = model_fn(x, params) where x is a
      PyTorch Tensor of image data, params is a list of PyTorch Tensors giving
      model weights, and scores is a PyTorch Tensor of shape (N, C) giving
      scores for the elements in x.
    - params: List of PyTorch Tensors giving weights for the model
    - learning_rate: Python scalar giving the learning rate to use for SGD

    Returns: Nothing
    """
    for t, (x, y) in enumerate(train_loader):
        # Move the data to the proper device (GPU or CPU)
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)

        # Forward pass: compute scores and loss
        scores = model_fn(x, params)
        loss = F.cross_entropy(scores, y)

        # Backward pass: PyTorch figures out which Tensors in the computational
        # graph has requires_grad=True and uses backpropagation to compute the
        # gradient of the loss with respect to these Tensors, and stores the
        # gradients in the .grad attribute of each Tensor.
        loss.backward()

        # Update parameters. We don't want to backpropagate through the
        # parameter updates, so we scope the updates under a torch.no_grad()
        # context manager to prevent a computational graph from being built.
        with torch.no_grad():
            for w in params:
                w -= learning_rate * w.grad

                # Manually zero the gradients after running the backward pass
                w.grad.zero_()

        if t % print_every == 0:
            print('Iteration %d, loss = %.4f' % (t, loss.item()))
            check_accuracy_part2(val_loader, model_fn, params)
            print()


### BareBones PyTorch: Train a Two-Layer Network
Now we are ready to run the training loop. We need to explicitly allocate tensors for the fully connected weights, `w1` and `w2`.

Each minibatch of CIFAR-10 has 64 examples, so the tensor shape is `[64, 3, 32, 32]`.

After flattening, `x` shape should be `[64, 3 * 32 * 32]`. This will be the size of the first dimension of `w1`.
The second dimension of `w1` is the hidden layer size, which will also be the first dimension of `w2`.

Finally, the output of the network is a 10-dimensional vector that represents the probability distribution over 10 classes.

No hyperparameter tuning is required for this baseline.

Train for one epoch..


In [ ]:
hidden_layer_size = 500
learning_rate = 1e-3

w1 = random_weight((3 * 32 * 32, hidden_layer_size))
w2 = random_weight((hidden_layer_size, 10))

train_part2(two_layer_fc, [w1, w2], learning_rate)


## Part III. Two-Layer Network with PyTorch Module API


Barebone PyTorch requires that we track all the parameter tensors by hand. This is fine for small networks with a few tensors, but it would be *extremely inconvenient* and *error-prone* to track tens or hundreds of tensors in larger networks.

PyTorch provides the `nn.Module` API for you to define arbitrary network architectures, while tracking every learnable parameters for you. In Part II, we implemented SGD ourselves. PyTorch also provides the `torch.optim` package that implements all the common optimizers, such as RMSProp, Adagrad, and Adam. You can refer to the [doc](http://pytorch.org/docs/master/optim.html) for the exact specifications of each optimizer.

To use the Module API, follow the steps below:

1. Subclass `nn.Module`. Give your network class an intuitive name like `TwoLayerFC`.

2. In the constructor `__init__()`, define all the layers you need as class attributes. Layer objects like `nn.Linear` are themselves `nn.Module` subclasses and contain learnable parameters, so that you don't have to instantiate the raw tensors yourself. `nn.Module` will track these internal parameters for you. Refer to the [doc](http://pytorch.org/docs/master/nn.html) to learn more about the dozens of builtin layers. **Warning**: don't forget to call the `super().__init__()` first!

3. In the `forward()` method, define the *connectivity* of your network. You should use the attributes defined in `__init__` as function calls that take tensor as input and output the "transformed" tensor. Do *not* create any new layers with learnable parameters in `forward()`! All of them must be declared upfront in `__init__`.

After you define your Module subclass, you can instantiate it as an object and call it just like the NN forward function in part II.


### Module API: Two-Layer Network


**Implementation note.**

This section defines a two-layer fully connected neural network using PyTorch's `nn.Module` API. The network should consist of two linear layers with a ReLU activation function in between, exactly as we did earlier using barebone PyTorch. This section fill in the missing parts of the provided code template to define the network's architecture and its forward pass.

Network Architecture:
- The network has one hidden layer.
- The input layer takes inputs of a specified size and feeds them into the hidden layer.
- The hidden layer should use ReLU as its activation function.
- The output layer produces scores for a specified number of classes.

Initialization:
- Initialize the weights of both linear layers using the Kaiming normalization method, which is suitable for layers followed by a ReLU activation function.

Forward Pass:
- The implementation should include a forward method that defines how data passes through the network.


In [ ]:
# an __init__() and a forward() method.                                      #
import torch
import torch.nn as nn
import torch.nn.init as init

class TwoLayerFC(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(TwoLayerFC, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU() # define ReLU activation as a layer
        self.fc2 = nn.Linear(hidden_dim, num_classes)

        self.apply(self._initialize_weights)

    def _initialize_weights(self, module):
        if isinstance(module, nn.Linear):
            init.xavier_uniform_(module.weight)

    def forward(self, x):
        x = flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        scores = x
        return scores


Validate the implementation.


In [ ]:
def test_TwoLayerFC():
    input_size = 50
    x = torch.zeros((64, input_size), dtype=dtype)  # minibatch size 64, feature dimension 50
    model = TwoLayerFC(input_size, 42, 10)
    scores = model(x)
    print(scores.size())  # expected shape [64, 10]

test_TwoLayerFC()


### Module API: Check Accuracy


Given the validation or test set, we can again check the classification accuracy of the neural network.

**Implementation note.**

This version is slightly different from the one in part II.

This section modify the existing function, `check_accuracy_part2`, which evaluates the accuracy of a classification model on a given dataset split (validation or test), to use PyTorch's object-oriented `Module API` instead. The original function takes a DataLoader, a model function, and a list of parameters as inputs to compute the model's accuracy. This section create a new function, `check_accuracy_part3`, that achieves the same objective using a PyTorch model instance.

Implementation details:
- Parameter Handling: Instead of passing model parameters as a separate list, your function should directly use a PyTorch model object, which encapsulates its parameters.

- Data Type for Labels: In the original function, labels `y` are converted to `torch.int64`. You should change this to `torch.long` to ensure compatibility with PyTorch loss functions and model outputs.

- Forward Pass: Modify the way the forward pass is called. Instead of using a separate function `model_fn`, you will directly call the forward method on the `model` object with the input tensor `x`.


In [ ]:
def check_accuracy_part3(loader, model):
    """
    Check the accuracy of a classification model.

    Inputs:
    - loader: A DataLoader for the data split we want to check
    - model: The model we choose

    Returns: Nothing, but prints the accuracy of the model
    """
    #split = 'val' if loader.dataset.train else 'test'
    split = 'val' if len(loader) == 79 else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)
            scores = model.forward(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))

        return acc


**Concept check.**

In `check_accuracy_part2` and hope in `check_accuracy_part3` too, the first line is hardcoded:

```python
split = 'val' if len(loader) == 79 else 'test'
```

 Can you explain why do we need this line and how exactly does it work?


**Answer.** If the number of vatches in the dataloader instance is 79 this would indicate that the loader corresponds to a validation dataset, otherwise a test set. This is because we've previously stated that the number of samples in the validation set is 5000 and the batch size is 64.


### Module API: Training Loop


We also use a slightly different training loop.

Rather than updating the values of the weights ourselves, we use an Optimizer object from the `torch.optim` package, which abstracts the notion of an optimization algorithm and provides implementations of most of the algorithms commonly used to optimize neural networks.


In [ ]:
def train_part3(model, optimizer, epochs=1):
    """
    Train a model on CIFAR-10 using the PyTorch Module API.

    Inputs:
    - model: A PyTorch Module giving the model to train.
    - optimizer: An Optimizer object we will use to train the model
    - epochs: (Optional) A Python integer giving the number of epochs to train for

    Returns: Nothing, but prints model accuracies during training.
    """
    epoch_loss = 0

    model = model.to(device=device)  # move the model parameters to CPU/GPU
    for e in range(epochs):
        for t, (x, y) in enumerate(train_loader):
            model.train()  # put model to training mode
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)

            scores = model(x)
            loss = F.cross_entropy(scores, y)

            epoch_loss += loss.item()

            # Zero out all of the gradients for the variables which the optimizer
            # will update.
            optimizer.zero_grad()

            # This is the backwards pass: compute the gradient of the loss with
            # respect to each  parameter of the model.
            loss.backward()

            # Actually update the parameters of the model using the gradients
            # computed by the backwards pass.
            optimizer.step()

            if t % print_every == 0:
                print('Iteration %d, loss = %.4f' % (t, loss.item()))
                _ = check_accuracy_part3(val_loader, model)
                print()

        epoch_loss /= train_loader.batch_size

        return epoch_loss


### Module API: Train a Two-Layer Network


Now we are ready to run the training loop. In contrast to part II, we don't explicitly allocate parameter tensors anymore.

Simply pass the input size, hidden layer size, and number of classes (i.e. output size) to the constructor of `TwoLayerFC`.

You also need to define an optimizer that tracks all the learnable parameters inside `TwoLayerFC`.

No hyperparameter tuning is required for this baseline.

Train for one epoch..


In [ ]:
hidden_layer_size = 500
learning_rate = 1e-3
model = TwoLayerFC(3 * 32 * 32, hidden_layer_size, 10)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

_ = train_part3(model, optimizer)


## Part IV. Two-Layer Network with PyTorch Sequential API


Part III introduced the PyTorch Module API, which allows you to define arbitrary learnable layers and their connectivity.

For simple models like a stack of feed forward layers, you still need to go through 3 steps: subclass `nn.Module`, assign layers to class attributes in `__init__`, and call each layer one by one in `forward()`. Is there a more convenient way?

Fortunately, PyTorch provides a container Module called `nn.Sequential`, which merges the above steps into one. It is not as flexible as `nn.Module`, because you cannot specify more complex topology than a feed-forward stack, but it's good enough for many use cases.


### Sequential API: Two-Layer Network


Let's see how to rewrite our two-layer fully connected network example with `nn.Sequential`, and train it using the training loop defined above.

Again, you don't need to tune any hyperparameters. Train for one epoch..


In [ ]:
# We need to wrap `flatten` function in a module in order to stack it
# in nn.Sequential
class Flatten(nn.Module):
    def forward(self, x):
        return flatten(x)

hidden_layer_size = 500
learning_rate = 1e-3

model = nn.Sequential(
    Flatten(),
    nn.Linear(3 * 32 * 32, hidden_layer_size),
    nn.ReLU(),
    nn.Linear(hidden_layer_size, 10),
)

# you can use Nesterov momentum in optim.SGD
optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)

train_part3(model, optimizer)


**Concept check.**

This time, expected observation: that the model achieves greater accuracy compared to the previous implementations.

What is the difference this time?

Once observed, explain how this difference might improve performance.


**Answer.** Accuracy improved thanks to  nesterov's momentum method. The core idea behind momentum is to work like a "memory" of the gradient updates, which accumulates constributions from past gradients, helping the algorithm to maintain a smooth gradient direction (reduces oscillations). When the direction of the gradients is the same for a long time, momentum accumulates constributions from past gradients and amplifies the magnitude of the gradient resulting in "bigger jumps ahead" in the parameter space.

Nesterov momentum provides additional benefits by "looking ahead" to where the parmeters are going to move if we applied the current momentum, and then computes the gradient based on that position. This results in faster and more stable convergence, especially in scenarios with noisy gradients.


### Sequential API: Experiment with Optimizers


**Implementation note.**

Given the baseline model implemented with PyTorch's Sequential API (as defined above), this section train this model for 1 epoch using as **optimizers**: `SGD with momentum`, `Adagrad`, `RMSProp`, `Adam`, `AdamW` and `NAdam`.

1. Refer to the PyTorch documentation about [optimizers](https://pytorch.org/docs/stable/optim.html.) to implement these optimizers.

2. Perform a learning rate search for each optimizer to find the learning rate that yields the best accuracy on the validation set.

For faster iteration:

- Use `hidden_layer_size = 50`, so that computations take less time.

- Utilize the `train_part3` function for training.

NOTE: You'll probably have to modify the `check_accuracy_part3()` function to return the computed accuracy (as a percentage).


In [ ]:
# for 1 epoch using different optimizers. Note to keep all the results.      #

optimizers = ['SGD', 'Adagrad', 'RMSprop', 'Adam', 'AdamW', 'NAdam']
learning_rates = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
hidden_layer_size = 50

best_lr_acc = {optimizer_name: (None, None) for optimizer_name in optimizers}

# Set random seed for reproducibility when re-initializing a new model
torch.manual_seed(42)

for optimizer_name in optimizers:

    best_loss = float('inf')
    best_lr = None
    best_model = None
    for lr in learning_rates:
        print(f'-------------------Optimizer: {optimizer_name}, Learning rate: {lr}-------------------')

        model = nn.Sequential(
            Flatten(),
            nn.Linear(3 * 32 * 32, hidden_layer_size),
            nn.ReLU(),
            nn.Linear(hidden_layer_size, 10),
        )

        # Initialize optimizer with the new learning rate
        if optimizer_name == 'SGD':
            optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True)
        elif optimizer_name == 'Adagrad':
            optimizer = optim.Adagrad(model.parameters(), lr=lr)
        elif optimizer_name == 'RMSprop':
            optimizer = optim.RMSprop(model.parameters(), lr=lr)
        elif optimizer_name == 'Adam':
            optimizer = optim.Adam(model.parameters(), lr=lr)
        elif optimizer_name == 'AdamW':
            optimizer = optim.AdamW(model.parameters(), lr=lr)
        elif optimizer_name == 'NAdam':
            optimizer = optim.NAdam(model.parameters(), lr=lr)

        epoch_loss = train_part3(model, optimizer, epochs=1)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_lr = lr
            best_model = model

    acc = check_accuracy_part3(val_loader, best_model)
    best_lr_acc[optimizer_name] = (best_lr, acc)

for optimizer_name, (best_lr, acc) in best_lr_acc.items():
    print(f'Optimizer: {optimizer_name}, Best Learning Rate: {best_lr}, Validation Accuracy: {acc}')


Keep the best classification results (based on validation accuracy) per optimizer in the following Table:


| Optimizer | best LR | Validation Accuracy |
|----------|----------|----------|
| SGD    | ...   | ...   |
| SGD + momentum    | 0.01   | 0.4258   |
| Adagrad    | 0.001   | 0.4116   |
| RMSProp    | 0.0001   | 0.4296   |
| Adam    | 0.001   | 0.4452   |
| AdamW    | 0.001   | 0.4464   |
| NAdam    | 0.001   | 0.3424   |


### Sequential API: Experiment with Activation Functions


**Implementation note.**

Using the optimizer and learning rate that performed the best in the previous question, train the same model for 1 epoch using different **activation functions**: `Leaky ReLU`, `ELU`, `GeLU`, `PReLU`, `SiLU` and `Mish`.

1. Refer to the PyTorch documentation about [non-linear activations](https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity) to implement these activation functions.

2. Observe the validation accuracy for each activation function and select the one that performs best.

For faster iteration, use again `hidden_layer_size = 50`, so that computations take less time.


In [ ]:
# for 1 epoch using different activations. Note to keep all the results.     #

best_lr = 0.001



activations = {
    "LeakyReLU": nn.LeakyReLU(negative_slope=0.01),
    "ELU": nn.ELU(alpha=1.0),
    "GELU": nn.GELU(),
    "PReLU": nn.PReLU(),  # Learnable parameter
    "SiLU": nn.SiLU(),    # Sigmoid-weighted Linear Unit (aka Swish)
    "Mish": nn.Mish()     # Mish activation
}

hidden_layer_size = 50

# Store losses for plotting
val_acc = {activation_name: None for activation_name in activations}

for activation_name, activation in activations.items():
    print(f'-------------------Activation: {activation_name}-------------------')

    model = nn.Sequential(
        Flatten(),
        nn.Linear(3 * 32 * 32, hidden_layer_size),
        activation,
        nn.Linear(hidden_layer_size, 10),
    )
    best_optimizer = optim.AdamW(model.parameters(), lr=best_lr)

    # Train the model for 1 epoch
    epoch_loss = train_part3(model, best_optimizer, epochs=1)
    acc = check_accuracy_part3(val_loader, model)

    val_acc[activation_name] = acc

for activation_name, acc in val_acc.items():
    print(f'Activation Function: {activation_name}, Validation Accuracy: {acc}')


Keep the classification results in the following Table:


| Activation function | Validation Accuracy |
|----------|----------|
| ReLU    | ...   |
| Leaky ReLU    | 0.434   |
| ELU    | 0.4402   |
| GeLU    | 0.4464   |
| PReLU    | 0.452   |
| SiLU    | 0.4494   |
| Mish    | 0.441   |


Από εδώ και κάτω δεν πρόλαβα :(


### Sequential API: Experiment with hidden layer sizes


**Implementation note.**

Using the optimizer, learning rate, and activation function that gave the best results earlier, experiment with different hidden layer sizes. Try at least the following values: `5`, `10`, `20`, `50`, `100`, `200`, `500`.

For each hidden layer size:

1. Train the model for 1 epoch.
2. Record the respective train and validation accuracy.

Then, after having performed all the experiments, plot a graph showing hidden layer sizes (x-axis) vs. train and validation accuracy (y-axis).


In [ ]:
# for 1 epoch using best optimizer, learning rate, and activation, adjusting #
# the hidden layer sizes.                                                                      #
best_lr = 0.001
best_optimizer = optim.AdamW(model.parameters(), lr=best_lr)
best_activation_f = None



hidden_layer_sizes = [5, 10, 20, 50, 100, 200, 500]
# Store losses for plotting
train_val_acc = {hidden_layer_size: (None, None) for hidden_layer_size in hidden_layer_sizes}

for hidden_layer_size in activations.keys():
    print(f'-------------------Hidden Size: {hidden_layer_size}-------------------')
    model = nn.Sequential(
        Flatten(),
        nn.Linear(3 * 32 * 32, hidden_layer_size),
        best_activation_f,
        nn.Linear(hidden_layer_size, 10),
    )

    # Train the model for 1 epoch and record the loss
    epoch_loss = train_part3(model, best_optimizer, epochs=1)
    val_acc = check_accuracy_part3(val_loader, model)
    train_acc = check_accuracy_part3(train_loader, model)

    train_val_acc[hidden_layer_size] = (train_acc, val_acc)


train_accuracies = [train_val_acc[size][0] for size in hidden_layer_sizes]
val_accuracies = [train_val_acc[size][1] for size in hidden_layer_sizes]

plt.figure(figsize=(10, 6))

plt.plot(hidden_layer_sizes, train_accuracies, label="Train Accuracy", marker='o', linestyle='-', color='b')
plt.plot(hidden_layer_sizes, val_accuracies, label="Validation Accuracy", marker='x', linestyle='--', color='r')

plt.title("Train vs Validation Accuracy with Different Hidden Layer Sizes", fontsize=16)
plt.xlabel("Hidden Layer Size", fontsize=14)
plt.ylabel("Accuracy", fontsize=14)

plt.legend()
plt.grid(True)
plt.show()


**Concept check.**

Observe the plot of hidden layer size vs. train and validation accuracy:

1.	Overfitting Analysis: Based on the trends in train and validation accuracy, identify if and where overfitting occurs. Explain how increasing the hidden layer size impacts the gap between train and validation accuracy, and why this might happen.

2.	Best Hidden Layer Size: Identify the hidden layer size that gave the highest validation accuracy, even if overfitting was present. Justify why this size might have been optimal for validation performance compared to smaller or larger sizes.

3.	Generalization Trade-off: Discuss the trade-off between model capacity (hidden layer size) and generalization to unseen data (validation accuracy). How does this trade-off affect your choice of the best hidden layer size for practical use?


**Answer.**
1. As the hidden layer size increases the model learns more complex patterns and fits the training data better. Therefore there training accuracy will increase.

2. As the hidden layer size increases, the validation acc might also improve however after a certain point we'll see a gap between the training and validation acc. This indicates overfitting (if train and val acc diverge too much). The model starts to overfit to the training data learning noise and irrelevant patterns.

3. The goal is for the model to learn complex patterns without overfitting. The larger the hidden layer size increases the capacity to learn those complex patterns but too much will make it overfit the training data and thus perform poorly on unseen data (bad generalization). The best hidden size is the one that offers the highest validation acc. That's why we use the validation set.


### Sequential API: Experiment with Dropout


**Implementation note.**

Overfitting can occur, especially as your model becomes more complex.

Experiment with [Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html) to regularize your model:

1.	Add a Dropout layer after the first fully connected layer in your model.

2.	Train the model with the best configuration for 1 epoch with dropout probabilities of `0.1`, `0.3`, `0.5`, `0.7`, `0.9`.

3.	Record the validation accuracy for each dropout probability.

4. Plot a graph showing dropout probability (x-axis) vs. train and validation accuracy (y-axis).


In [ ]:
# after the first fully connected layer. Train the model for 1 epoch using   #
# the best optimizer, learning rate, and activation, with dropout rates of   #
# 0.1, 0.3, 0.5, 0.7, and 0.9. Record validation accuracy for each rate.     #


**Concept check.**

Observe the plot of dropout probability vs. train and validation accuracy:

1.	Overfitting Reduction: Discuss how dropout affected overfitting in the model with the best configuration. Did adding dropout successfully reduce the gap between train and validation accuracy? If yes, at what dropout probability was this effect most noticeable?

2.	Best Dropout Probability: Identify the dropout probability that gave the highest validation accuracy. Why do you think this value worked best?

3.	Too Much Dropout: Notice the trend for very high dropout probabilities (e.g., 0.7, 0.9). How does excessive regularization impact train and validation accuracy, and why does this happen?

4.	Generalization: Based on your observations, explain how dropout helps improve the generalization of the model while preventing overfitting.


### Sequential API: Experiment with Learning Rate Schedulers


**Implementation note.**

Learning rate schedulers adjust the learning rate during training and can help the model converge more effectively. Using the best configuration so far (optimizer, learning rate, activation function, and hidden layer size), experiment with learning rate schedulers:

- `StepLR`
- `LinearLR`
- `CosineAnnealingLR`
- `ExponentialLR`
- `ReduceLROnPlateau`

1. Refer to the PyTorch [scheduler documentation](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate) to implement these schedulers.

2.	Train the model for 5 epochs with each scheduler.

3.	Observe how the validation accuracy evolves over the epochs and determine which scheduler performs best.


In [ ]:
# for 5 epochs using the best optimizer, learning rate, and activation. Add  #
# a learning rate scheduler (`StepLR`, `LinearLR`, etc.).                    #
# to the optimizer, and experiment with its impact on validation accuracy.   #


### Sequential API: Evaluate the final model on the test set of CIFAR-10


**Implementation note.**

Using the best configuration found during the previous experiments (optimizer, learning rate, activation function, hidden layer size, dropout rate, and learning rate scheduler), train the model for 5 epoch and evaluate its performance on the test set.

1.	Report the test accuracy.

2.	Compare it with the validation accuracy to assess if your model generalized well.


In [ ]:
# optimizer, learning rate, activation function, hidden layer size, and      #
# dropout rate. Evaluate the trained model on the test set, and compare its  #
# performance with validation accuracy.                                      #


## Technical report


**Concept check.**

After completing all experiments, summarize your findings in a concise and short technical report. Include the baseline accuracy (SGD with lr=1e-3 and ReLU), compare the performance of different optimizers, activation functions, hidden layer sizes, dropout rates, and learning rate schedulers, and highlight the best configuration. Finally, report the test accuracy of your final model, discuss its generalization performance, and explain why the chosen configuration might have worked best.
